# RAG Ablation Study: Baseline Dense Retrieval vs. Cross-Encoder Re-Ranking

**Corpus:** ITF *Rules of Tennis*, 2026 edition (single PDF).

Two pipelines are compared over the **same chunked corpus**:

| Pipeline | Flow |
|---|---|
| **Baseline** | Bi-Encoder → FAISS Top-10 → take Top-3 → LLM → Answer 1 |
| **Re-ranking** | Bi-Encoder → FAISS Top-10 → Cross-Encoder re-sort → Top-3 → LLM → Answer 2 |

The only variable is the ranking stage: retrieval, chunking, and the LLM prompt are identical.
Everything is configured in `config.yaml` — no code changes needed to retune.

## Section 1: Setup & Environment

In [2]:
"""Section 1 — imports, seeds, configuration, secrets."""
import json
import os
import random
import re
import statistics
import time
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from dotenv import load_dotenv

# Determinism: fix all seeds (scaling experiment uses a synthetic random index)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

WORKSPACE = Path.cwd()
if not (WORKSPACE / "config.yaml").exists():  # fall back to this notebook's folder
    WORKSPACE = Path("/home/tomato/Desktop/RAG-ablation")

CFG = yaml.safe_load((WORKSPACE / "config.yaml").read_text())

load_dotenv(WORKSPACE / ".env")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")

print("Workspace :", WORKSPACE)
print("PDF       :", CFG["data"]["pdf_path"])
print("Models    :", CFG["models"]["bi_encoder"], "|", CFG["models"]["cross_encoder"])
print("Device    :", CFG["models"]["device"])
print("API key   :", "present" if GEMINI_API_KEY else "MISSING — LLM cells will use placeholders")

Workspace : /home/tomato/Desktop/RAG-ablation
PDF       : 2026-rules-of-tennis-english.pdf
Models    : sentence-transformers/all-MiniLM-L6-v2 | cross-encoder/ms-marco-MiniLM-L-6-v2
Device    : cpu
API key   : MISSING — LLM cells will use placeholders


## Section 2: PDF Loading & Cleaning

Extract raw text with `pypdf`, then strip PDF-layout artifacts that would pollute
embeddings: page numbers on their own line, soft-wrapped lines, unicode oddities,
and inconsistent whitespace. Text is kept **in document order** so section headers
can be detected positionally in Section 3.

In [3]:
from pypdf import PdfReader

PDF_PATH = WORKSPACE / CFG["data"]["pdf_path"]
assert PDF_PATH.exists(), f"Corpus PDF not found: {PDF_PATH}"

reader = PdfReader(str(PDF_PATH))
raw_pages = [(p.extract_text() or "") for p in reader.pages]
print(f"Extracted {len(raw_pages)} pages, {sum(len(p) for p in raw_pages):,} raw characters.")

Extracted 47 pages, 83,838 raw characters.


In [4]:
def clean_page(text: str) -> str:
    """Normalise one page of extracted PDF text.

    - Strip stray page-number lines (e.g. ' 7 ', ' 3 \n ').
    - Normalise unicode dashes/quotes to ASCII equivalents.
    - Collapse runs of spaces/tabs; keep single newlines between paragraphs.
    """
    # Unicode normalisation (keeps regexes simple downstream)
    text = (
        text.replace("\u2018", "'").replace("\u2019", "'")
        .replace("\u201c", '"').replace("\u201d", '"')
        .replace("\u2013", "-").replace("\u2014", "-")
        .replace("\u00a0", " ")
    )
    out_lines = []
    for line in text.split("\n"):
        stripped = line.strip()
        if re.fullmatch(r"\d{1,3}", stripped):      # bare page number → drop
            continue
        if re.fullmatch(r"", stripped):              # blank line → paragraph break marker
            out_lines.append("")
            continue
        out_lines.append(re.sub(r"[ \t]+", " ", stripped))
    # Collapse multiple blank lines into one
    joined = "\n".join(out_lines)
    joined = re.sub(r"\n{3,}", "\n\n", joined)
    return joined.strip()


cleaned_pages = [clean_page(p) for p in raw_pages]
full_text = "\n\n".join(p for p in cleaned_pages if p)
print(f"Cleaned corpus: {len(full_text):,} characters.")
print("\n" + full_text[:400] + "\n...")

Cleaned corpus: 81,568 characters.

ITF RULES OF TENNIS
 6

All the rules,
on the go.
Available to
download now.
RULES OF
TENNIS

CONTENTS

FOREWORD 1
Rule 1 THE COURT 2
Rule 2 PERMANENT FIXTURES 3
Rule 3 THE BALL 3
Rule 4 THE RACKET 4
Rule 5 SCORE IN A GAME 5
Rule 6 SCORE IN A SET 6
Rule 7 SCORE IN A MATCH 6
Rule 8 SERVER AND RECEIVER 6
Rule 9 CHOICE OF ENDS AND SERVICE 6
Rule 10 CHANGE OF ENDS 7
Rule 11 BALL IN PLAY 7
Rule 12 BAL
...


## Section 3: Section Detection & Chunking

Blind token-window chunking would split a rule from its exception — fatal for this
experiment, since the failure modes depend on *cases* being retrievable in isolation.
Strategy (see design doc §3):

1. **Section split** — detect `1. THE COURT` … `31. PLAYER ANALYSIS TECHNOLOGY`,
   `APPENDIX I`–`XII`, and named addenda (Rules of Wheelchair Tennis, Beach Tennis…).
2. **Block split** — within each section, split at `Case N:` / `Note:` boundaries.
3. **Merge** — blocks shorter than `min_chunk_words` merge into the previous chunk.
4. **Oversize split** — long blocks split on sentence boundaries with overlap.

Every chunk carries a human-readable title (`Rule 4 — Case 4: ...`) used for reporting.

In [5]:
# --- 3.1 Section detection -------------------------------------------------
# The main body has exactly 31 numbered rules; appendices start after them.
# We locate the LAST occurrence of each header to skip any table-of-contents hits.

NUM_RULES = 31

def find_header_positions(text: str) -> list[tuple[int, str]]:
    """Return (start_offset, title) for every structural header, in order."""
    positions: list[tuple[int, str]] = []

    # Numbered rules: 'N. ALL-CAPS TITLE' (title may wrap onto the next line)
    nums = "|".join(str(n) for n in range(1, NUM_RULES + 1))
    rule_pat = re.compile(
        rf"(?m)^[ \t]*({nums})\."
        r"[ \t]+([A-Z][A-Z ,\'\-“”()/&]+?)[ \t]*(?:\n|$)"
    )
    last_by_num: dict[str, tuple[int, str]] = {}
    for m in rule_pat.finditer(text):
        num, title = m.group(1), m.group(2).strip()
        if len(title) < 3:
            continue
        last_by_num[num] = (m.start(), f"Rule {num} — {title.title()}")
    for num in range(1, NUM_RULES + 1):
        if str(num) in last_by_num:
            positions.append(last_by_num[str(num)])

    # Appendices: 'APPENDIX VII' optionally followed by an uppercase title line
    app_pat = re.compile(
        r"(?m)^[ \t]*APPENDIX[ \t]+([IVXLCDM]+)[ \t]*(?:\n[ \t]*([A-Z][A-Z ,\-']+))?"
    )
    apps: dict[str, tuple[int, str]] = {}
    for m in app_pat.finditer(text):
        numeral = m.group(1)
        title = (m.group(2) or "").strip().title()
        label = f"Appendix {numeral}" + (f" — {title}" if title else "")
        apps[numeral] = (m.start(), label)
    positions.extend(apps.values())

    # Named addenda between the rules and the appendices
    for name in ("RULES OF WHEELCHAIR TENNIS", "SHORT SETS"):
        m = None
        for m in re.finditer(rf"(?m)^[ \t]*{name}[ \t]*(?:\n|$)", text):
            pass  # keep the last match
        if m:
            positions.append((m.start(), name.title()))

    positions.sort(key=lambda t: t[0])
    return positions


header_positions = find_header_positions(full_text)
sections = []
for i, (start, title) in enumerate(header_positions):
    end = header_positions[i + 1][0] if i + 1 < len(header_positions) else len(full_text)
    body = full_text[start:end].strip()
    sections.append({"title": title, "body": body})

print(f"Detected {len(sections)} sections:")
for s in sections:
    print(f"  {s['title']:<55} ({len(s['body']):>6,} chars)")

Detected 44 sections:
  Rule 18 — Foot Fault                                    (   725 chars)
  Rule 19 — Service Fault                                 (   846 chars)
  Rule 20 — Second Service                                (   211 chars)
  Rule 21 — When To Serve And Receive                     (   412 chars)
  Rule 22 — The Let During A Service                      (   564 chars)
  Rule 23 — The Let                                       (   387 chars)
  Rule 24 — Player Loses Point                            ( 3,425 chars)
  Rule 25 — A Good Return                                 ( 1,639 chars)
  Rule 26 — Hindrance                                     ( 1,242 chars)
  Rule 27 — Correcting Errors                             ( 3,947 chars)
  Rule 28 — Role Of Court Officials                       (   133 chars)
  Rule 29 — Continuous Play                               ( 1,844 chars)
  Rule 30 — Coaching                                      (   766 chars)
  Rule 31 — Player Analysis T

In [6]:
# --- 3.2 Block split, merge, oversize split --------------------------------

CHUNK_CFG = CFG["chunking"]
MAX_WORDS = CHUNK_CFG["chunk_size_words"]
OVERLAP_WORDS = CHUNK_CFG["chunk_overlap_words"]
MIN_WORDS = CHUNK_CFG["min_chunk_words"]

def wc(s: str) -> int:
    return len(s.split())

def split_sentences(block: str) -> list[str]:
    """Split on sentence boundaries (handles abbreviations like 'Rule 4.' poorly but fine here)."""
    parts = re.split(r"(?<=[.!?:])\s+", block)
    return [p.strip() for p in parts if p.strip()]

def split_oversized(block: str) -> list[str]:
    """Sentence-boundary split with word-overlap for blocks longer than MAX_WORDS."""
    sents = split_sentences(block)
    pieces, cur = [], []
    cur_w = 0
    for sent in sents:
        sw = wc(sent)
        if cur and cur_w + sw > MAX_WORDS:
            pieces.append(" ".join(cur))
            tail = cur[-OVERLAP_WORDS:] if OVERLAP_WORDS else []
            cur, cur_w = list(tail), sum(wc(t) for t in tail)
        cur.append(sent)
        cur_w += sw
    if cur:
        pieces.append(" ".join(cur))
    return pieces

def title_from_block(section_title: str, block: str) -> str:
    """Build a readable chunk title, e.g. 'Rule 4 — The Racket — Case 4'."""
    m = re.match(r"^(Case\s+\d+|Note)\s*:", block)
    if m:
        return f"{section_title} — {m.group(1)}"
    # For plain blocks, append the first few words as a hint
    hint = " ".join(block.split()[:6]).rstrip(":.,").title()
    return f"{section_title} — {hint}…"

def build_chunks(sections: list[dict]) -> list[dict]:
    chunks: list[dict] = []
    for sec in sections:
        title, body = sec["title"], sec["body"]
        # Remove the header line itself from the body we embed
        body_wo_header = re.sub(r"^.*?(?:\n|$)", "", body, count=1).strip()
        if not body_wo_header:
            continue
        # Split at Case/Note boundaries (keep the marker with its block)
        blocks = re.split(r"(?=^(?:Case\s+\d+|Note)\s*:)", body_wo_header, flags=re.M)
        blocks = [b.strip() for b in blocks if b.strip()]
        # Merge short blocks forward
        merged: list[str] = []
        for b in blocks:
            if merged and wc(b) < MIN_WORDS and wc(merged[-1]) + wc(b) <= MAX_WORDS:
                merged[-1] = merged[-1] + "\n" + b
            else:
                merged.append(b)
        # Oversize split
        for b in merged:
            for piece in ([b] if wc(b) <= MAX_WORDS else split_oversized(b)):
                if wc(piece) < 5:
                    continue
                chunks.append({
                    "id": len(chunks),
                    "section": title,
                    "title": title_from_block(title, piece),
                    "text": piece,
                    "words": wc(piece),
                })
    return chunks

chunks = build_chunks(sections)
print(f"Built {len(chunks)} chunks "
      f"(mean {statistics.mean(c['words'] for c in chunks):.0f} words, "
      f"max {max(c['words'] for c in chunks)}).\n")
pd.DataFrame([{k: c[k] for k in ('id','section','title','words')} for c in chunks]).head(12)

Built 305 chunks (mean 293 words, max 785).



,id,section,title,words
0,0,Rule 18 — Foot Fault,Rule 18 — Foot Fault — During The Service Moti...,73
1,1,Rule 18 — Foot Fault,Rule 18 — Foot Fault — Case 1,52
2,2,Rule 19 — Service Fault,Rule 19 — Service Fault — The Service Is A Fau...,68
3,3,Rule 19 — Service Fault,Rule 19 — Service Fault — Case 1,56
4,4,Rule 19 — Service Fault,Rule 19 — Service Fault — Case 2,36
5,5,Rule 20 — Second Service,Rule 20 — Second Service — If The First Servic...,36
6,6,Rule 21 — When To Serve And Receive,Rule 21 — When To Serve And Receive — The Serv...,68
7,7,Rule 22 — The Let During A Service,Rule 22 — The Let During A Service — The Servi...,96
8,8,Rule 23 — The Let,Rule 23 — The Let — In All Cases When A Let…,25
9,9,Rule 23 — The Let,Rule 23 — The Let — Case 1,45


## Section 4: Bi-Encoder Index (FAISS)

`all-MiniLM-L6-v2` produces 384-d embeddings; we L2-normalise them so FAISS
`IndexFlatIP` (exact inner product) computes cosine similarity exactly.
Exact search → perfect recall → **ranking is the only variable** under study.

In [7]:
import faiss
from sentence_transformers import CrossEncoder, SentenceTransformer

DEVICE = CFG["models"]["device"] or None

encoder = SentenceTransformer(CFG["models"]["bi_encoder"], device=DEVICE)
cross_encoder = CrossEncoder(CFG["models"]["cross_encoder"], device=DEVICE)

# Embed titles + bodies: the title disambiguates 'Rule 18 — Case 1' style chunks
corpus_texts = [f"{c['title']}\n{c['text']}" for c in chunks]
t0 = time.perf_counter()
emb = encoder.encode(corpus_texts, batch_size=32, normalize_embeddings=True,
                     show_progress_bar=False)
emb = np.ascontiguousarray(emb.astype(np.float32))
print(f"Encoded {emb.shape[0]} chunks × {emb.shape[1]} dims in {time.perf_counter()-t0:.1f}s")

index = faiss.IndexFlatIP(emb.shape[1])
index.add(emb)
print(f"FAISS index: {index.ntotal} vectors, space={index.metric_type} (0=InnerProduct)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

KeyboardInterrupt: 

## Section 5: Retrieval Pipelines

- **Baseline**: encode query → FAISS Top-10 → take first `top_k_final`.
- **Re-ranking**: same Top-10 candidates → Cross-Encoder scores every
  `(query, chunk)` pair → re-sort → take Top-3.

Both return the candidate ids with ranks so Section 7 can measure rank shifts.

In [ ]:
TOP_K_INIT = CFG["retrieval"]["top_k_initial"]
TOP_K_FINAL = CFG["retrieval"]["top_k_final"]

def retrieve(query: str) -> dict:
    """Run both pipelines on one query; return ranked results for analysis/profiling."""
    q_vec = np.ascontiguousarray(
        encoder.encode([query], normalize_embeddings=True).astype(np.float32))
    dists, ids = index.search(q_vec, TOP_K_INIT)
    cand_ids = [int(i) for i in ids[0] if i >= 0]
    cand_scores = [float(d) for d in dists[0] if d >= -1e20]

    # Pipeline 1 — baseline: dense order as-is
    base_rank = cand_ids[:TOP_K_FINAL]

    # Pipeline 2 — cross-encoder re-rank over all candidates
    pairs = [(query, corpus_texts[i]) for i in cand_ids]
    ce_scores = cross_encoder.predict(pairs)
    order = np.argsort(-np.asarray(ce_scores))
    reranked_ids = [cand_ids[o] for o in order]
    rr_rank = reranked_ids[:TOP_K_FINAL]

    return {
        "candidates": cand_ids,
        "dense_scores": cand_scores,
        "ce_scores": [float(ce_scores[o]) for o in order],
        "reranked": reranked_ids,
        "baseline_top": base_rank,
        "rerank_top": rr_rank,
    }

def top_chunks(res: dict, which: str) -> list[dict]:
    ids = res[f"{which}_top"]
    return [chunks[i] for i in ids]

# Quick smoke test
demo = retrieve("How many court sides are used in doubles?")
print("Baseline  :", [chunks[i]['title'][:48] for i in demo['baseline_top']])
print("Re-ranked :", [chunks[i]['title'][:48] for i in demo['rerank_top']])

## Section 6: LLM Answer Generation (Gemini)

Identical prompt template for both pipelines — only the retrieved context differs.
`temperature=0` for reproducible answers. Without `GEMINI_API_KEY`, cells produce a
clearly-labelled placeholder so the notebook still runs end-to-end.

In [ ]:
PROMPT_TEMPLATE = """You are a tennis rules assistant. Answer strictly using ONLY the provided \
excerpts from the ITF Rules of Tennis. If the excerpts do not contain the answer, say so.

EXCERPTS:
{context}

QUESTION: {question}

Answer concisely (2-5 sentences), citing the rule/case you rely on."""

_gemini_model = None
if GEMINI_API_KEY:
    import google.generativeai as genai
    genai.configure(api_key=GEMINI_API_KEY)
    _gemini_model = genai.GenerativeModel(CFG["llm"]["model"])

def build_prompt(question: str, docs: list[dict]) -> str:
    ctx = "\n\n".join(f"[{d['title']}]\n{d['text']}" for d in docs)
    return PROMPT_TEMPLATE.format(context=ctx, question=question)

def ask_llm(question: str, docs: list[dict]) -> tuple[str, float]:
    """Return (answer, latency_seconds). Placeholder when no API key."""
    prompt = build_prompt(question, docs)
    if _gemini_model is None:
        titles = "; ".join(d["title"] for d in docs)
        return (f"[PLACEHOLDER — no GEMINI_API_KEY] Would answer using: {titles}", float("nan"))
    t0 = time.perf_counter()
    resp = _gemini_model.generate_content(
        prompt,
        generation_config={"temperature": CFG["llm"]["temperature"],
                           "max_output_tokens": CFG["llm"]["max_output_tokens"]},
    )
    return resp.text.strip(), time.perf_counter() - t0

## Section 7: Failure-Mode Analysis

Eight queries designed around known dense-retrieval weaknesses (keyword traps,
exceptions, negations, buried chunks). For each query we declare the **target chunk**
by a distinctive phrase; we then record where it ranked under dense retrieval vs.
cross-encoder re-ranking, and generate both LLM answers.

In [ ]:
# Each target is identified by a distinctive substring of the correct chunk.
QUERIES = [
    {"mode": "Keyword trap", "query": "Can a player keep using a racket after the strings break during a point?",
     "target": "strings break"},
    {"mode": "Keyword trap", "query": "Is a bird hitting the ball in play considered a hindrance?",
     "target": "bird"},
    {"mode": "Exception", "query": "If a let is called on a second service, does the whole point get replayed?",
     "target": "only the part of the rally following the service is replayed"},
    {"mode": "Exception", "query": "Is a ball that is soft but not broken at the end of a point replayed?",
     "target": "does not lose its shape"},
    {"mode": "Negation", "query": "In a singles match, may the server stand behind the part of the baseline between the singles and doubles sidelines?",
     "target": "between the imaginary extension of the singles sideline and the doubles sideline"},
    {"mode": "Negation", "query": "Is it a foot fault if the server has both feet off the ground while serving?",
     "target": "both feet momentarily off the ground"},
    {"mode": "Buried chunk", "query": "How many times may the ball bounce before a wheelchair tennis player loses the point?",
     "target": "allowed two bounces"},
    {"mode": "Buried chunk", "query": "How many unsuccessful Electronic Review appeals is a player allowed per set?",
     "target": "three (3) unsuccessful appeals"},
]

def resolve_target(sub: str) -> int | None:
    """Find the chunk id whose text contains `sub` (case-insensitive)."""
    sub_l = sub.lower()
    hits = [c["id"] for c in chunks if sub_l in c["text"].lower()]
    return hits[0] if hits else None

missing = [q for q in QUERIES if resolve_target(q["target"]) is None]
print("Queries with unresolved targets:", len(missing))
for q in missing:
    print("  ", q["target"])

In [ ]:
rows = []
answers = []
for q in QUERIES:
    tid = resolve_target(q["target"])
    res = retrieve(q["query"])
    cand = res["candidates"]
    dense_rank = cand.index(tid) + 1 if tid in cand else None      # 1-based, None = outside Top-10
    rr_list = res["reranked"]
    rr_rank = rr_list.index(tid) + 1 if tid in rr_list else None
    rows.append({
        "mode": q["mode"], "query": q["query"], "target_chunk": chunks[tid]["title"],
        "dense_rank": dense_rank, "rerank_rank": rr_rank,
        "in_baseline_top3": tid in res["baseline_top"],
        "in_rerank_top3": tid in res["rerank_top"],
    })
    # LLM answers for both pipelines
    a1, l1 = ask_llm(q["query"], top_chunks(res, "baseline"))
    a2, l2 = ask_llm(q["query"], top_chunks(res, "rerank"))
    answers.append({"query": q["query"], "answer_baseline": a1, "answer_rerank": a2,
                    "llm_latency_baseline_s": l1, "llm_latency_rerank_s": l2})

rank_df = pd.DataFrame(rows)
ans_df = pd.DataFrame(answers)
display(rank_df[["mode","query","target_chunk","dense_rank","rerank_rank","in_baseline_top3","in_rerank_top3"]])

In [ ]:
n_fail_base = (~rank_df["in_baseline_top3"]).sum()
n_fixed = ((~rank_df["in_baseline_top3"]) & rank_df["in_rerank_top3"]).sum()
print(f"Baseline misses target in Top-{TOP_K_FINAL}: {n_fail_base}/{len(rank_df)}")
print(f"Re-ranking recovers:                          {n_fixed}/{len(rank_df)}")
print("\nRank-shift summary (dense → re-ranked):")
for _, r in rank_df.iterrows():
    arrow = f"{r['dense_rank']} → {r['rerank_rank']}" if r["dense_rank"] else "outside Top-10"
    flag = "RESCUED" if (not r["in_baseline_top3"]) and r["in_rerank_top3"] else (
        "lost" if r["in_baseline_top3"] and not r["in_rerank_top3"] else "ok")
    print(f"  [{flag:>7}] {r['query'][:60]:<62} {arrow}")

In [ ]:
# Show the paired LLM answers for the queries where ranking actually changed
changed = rank_df[rank_df["in_baseline_top3"] != rank_df["in_rerank_top3"]]["query"].tolist()
for a in ans_df:
    if a["query"] not in changed:
        continue
    display(Markdown(f"### Q: {a['query']}\n\n**Answer 1 (baseline):** {a['answer_baseline']}\n\n"
                     f"**Answer 2 (re-ranked):** {a['answer_rerank']}"))

## Section 8: Latency Profiling

Timed separately: bi-encoder query encode, FAISS search, Cross-Encoder scoring of the
10 candidates, and full retrieval pipeline. Warmup excluded; mean/median/p95/std reported.
LLM latency is reported separately (network-bound, not comparable to local inference).

In [ ]:
PROF = CFG["profiling"]
WARMUPS, RUNS = PROF["warmup_runs"], PROF["timed_runs"]
profile_query = "Is a bird hitting the ball in play considered a hindrance?"

def timed(fn) -> list[float]:
    """Return per-run wall-clock seconds for fn(), warmups discarded."""
    for _ in range(WARMUPS):
        fn()
    ts = []
    for _ in range(RUNS):
        t0 = time.perf_counter()
        fn()
        ts.append(time.perf_counter() - t0)
    return ts

q_vec = np.ascontiguousarray(encoder.encode([profile_query], normalize_embeddings=True).astype(np.float32))
cand_texts = [corpus_texts[i] for i in retrieve(profile_query)["candidates"]]
pairs = [(profile_query, t) for t in cand_texts]

samples = {
    "Bi-Encoder query encode": timed(lambda: encoder.encode([profile_query], normalize_embeddings=True)),
    "FAISS Top-10 search": timed(lambda: index.search(q_vec, TOP_K_INIT)),
    f"Cross-Encoder scoring ({TOP_K_INIT} pairs)": timed(lambda: cross_encoder.predict(pairs)),
    "Full retrieval pipeline": timed(lambda: retrieve(profile_query)),
}

lat_rows = []
for name, ts in samples.items():
    arr = np.array(ts) * 1000.0  # ms
    lat_rows.append({"stage": name, "mean_ms": arr.mean(), "median_ms": np.median(arr),
                     "p95_ms": np.percentile(arr, 95), "std_ms": arr.std(ddof=1)})
latency_df = pd.DataFrame(lat_rows).round(2)
display(latency_df)

llm_lat = ans_df[["llm_latency_baseline_s", "llm_latency_rerank_s"]].dropna()
if len(llm_lat):
    print(f"LLM round-trip (reported separately, network-bound): "
          f"mean {(llm_lat.mean().mean()):.2f}s over {len(llm_lat)*2} calls")
else:
    print("LLM latency: not measured (no API key — placeholders used).")

## Section 9: Scaling Analysis

*Why not run the Cross-Encoder over the entire corpus?* CE cost is **O(N)** per query
(every pair must pass through the transformer), while FAISS `IndexFlatIP` is exact
but cheap per-vector and ANN indexes are ~O(log N). We measure real CE cost per pair,
extrapolate to 1K/10K/100K chunks, and compare against FAISS search on a synthetic
100K-vector index.

In [ ]:
# 1) Measure CE cost per (query, chunk) pair on the actual corpus
sample_pairs = [(profile_query, corpus_texts[i]) for i in range(min(50, len(chunks)))]
t_ce_50 = min(timed(lambda: cross_encoder.predict(sample_pairs)))  # best-of gives stable estimate
per_pair_ms = (t_ce_50 / len(sample_pairs)) * 1000.0
print(f"CE: {per_pair_ms:.2f} ms per (query, chunk) pair (batch of {len(sample_pairs)}).")

# 2) Extrapolate to scaling targets
scale_rows = []
for n in PROF["scaling_targets"]:
    scale_rows.append({"corpus_chunks": n,
                       "ce_full_scan_min": (per_pair_ms * n) / 60000.0,
                       "ce_top10_ms": per_pair_ms * TOP_K_INIT})
scale_df = pd.DataFrame(scale_rows)

# 3) FAISS comparison on a synthetic 100K-vector index (same dims)
N_SYNTH = max(PROF["scaling_targets"])
rng = np.random.default_rng(SEED)
synth = rng.standard_normal((N_SYNTH, emb.shape[1]), dtype=np.float32)
faiss.normalize_L2(synth)
synth_index = faiss.IndexFlatIP(synth.shape[1])
synth_index.add(synth)
t_faiss = timed(lambda: synth_index.search(q_vec, TOP_K_INIT))
print(f"FAISS search over {N_SYNTH:,} synthetic vectors: {np.median(t_faiss)*1000:.2f} ms (median)")

scale_df["faiss_search_ms"] = np.median(t_faiss) * 1000.0
scale_df["speedup_vs_full_scan"] = (scale_df["ce_full_scan_min"] * 60000.0) / scale_df["faiss_search_ms"]
display(scale_df.round(2))
print("\nTakeaway: full-corpus CE scales linearly (minutes at 100K chunks); "
      "retrieve-then-rerank keeps CE at a constant ~Top-10 pairs (~ms) regardless of corpus size.")

## Section 10: Export Results

Persist the evidence tables to `results/` for the written report.

In [ ]:
RESULTS = WORKSPACE / "results"
RESULTS.mkdir(exist_ok=True)

rank_df.to_csv(RESULTS / "rank_shifts.csv", index=False)
ans_df.to_csv(RESULTS / "llm_answers.csv", index=False)
latency_df.to_csv(RESULTS / "latency.csv", index=False)
scale_df.to_csv(RESULTS / "scaling.csv", index=False)
pd.DataFrame(chunks).to_json(RESULTS / "chunks.json", orient="records", force_ascii=False, indent=2)

summary = {
    "n_chunks": len(chunks),
    "top_k_initial": TOP_K_INIT, "top_k_final": TOP_K_FINAL,
    "baseline_misses": int(n_fail_base), "rerank_rescues": int(n_fixed),
    "total_queries": len(rank_df),
    "ce_per_pair_ms": round(per_pair_ms, 3),
    "api_key_present": bool(GEMINI_API_KEY),
}
(RESULTS / "summary.json").write_text(json.dumps(summary, indent=2))
print("Exported to", RESULTS)
display(pd.DataFrame([summary]).T)